# MARS-SOHO Phase 1C — train-only reconstruction fidelity

This notebook diagnoses the distribution model before another continual run. It compares empirical replay, ambient spherical replay, and class-specific tangent low-rank reconstruction on CIFAR-100 training splits only. It never extracts or opens test features.

In [ ]:
# === Locked study settings; do not edit during this run. ===
REPO_GIT_URL = 'https://github.com/ZaPhat206/SOHO-CL.git'
REPO_BRANCH = 'experiment/soho-selfcontained'
DATASET_KEY = 'cifar100'
WORK_DIR = '/content/SOHO-CL'
FEATURE_CACHE_ROOT = '/content/mars_soho_phase1_features'
OUTPUT_ROOT = '/content/mars_soho_phase1c_outputs'
BATCH_SIZE = 128
NUM_WORKERS = 2
EXPECTED_CONFIG_SHA256 = '9ca4940b50d7ebc7560ce65e0ddcf00e8ead47adb5e0d626fbd1fac0e7461da0'
EXPECTED_RUNNER_SHA256 = 'ce4f22a9bbd5075484303879a6bffeff64eea5b3750ea1772b20e72ace517548'

In [ ]:
# Fresh checkout, dependencies and immutable source verification.
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
os.chdir('/content')
repo=Path(WORK_DIR)
if repo.exists(): shutil.rmtree(repo)
subprocess.run(['git','clone','--branch',REPO_BRANCH,'--single-branch',REPO_GIT_URL,WORK_DIR],check=True)
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements-kaggle.txt','kagglehub','huggingface_hub','pandas','matplotlib','seaborn'],check=True)
import torch
assert torch.cuda.is_available(),'Select Runtime -> Change runtime type -> T4 GPU.'
def sha(path): return hashlib.sha256(Path(path).read_bytes()).hexdigest()
CONFIG='configs/mars_soho_phase1c_fidelity_train_only.json'
RUNNER='tools/mars_soho_phase1c.py'
assert sha(CONFIG)==EXPECTED_CONFIG_SHA256,'Config hash mismatch'
assert sha(RUNNER)==EXPECTED_RUNNER_SHA256,'Runner hash mismatch'
assert not subprocess.check_output(['git','status','--porcelain'],text=True).strip()
print('GPU:',torch.cuda.get_device_name(0))
print('commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())
print('MARS-SOHO PHASE-1C SOURCE CHECK: PASS')

In [ ]:
# Download verified ViT checkpoint and CIFAR-100.
import kagglehub
from huggingface_hub import hf_hub_download
CHECKPOINT_PATH=hf_hub_download(repo_id='timm/vit_base_patch16_224.augreg2_in21k_ft_in1k',filename='model.safetensors')
assert Path(CHECKPOINT_PATH).stat().st_size==346284714
assert sha(CHECKPOINT_PATH)=='32aa17d6e17b43500f531d5f6dc9bc93e56ed8841b8a75682e1bb295d722405b'
DATASET_ROOT=kagglehub.dataset_download('zaphat206/cifar-100')
print('dataset:',DATASET_KEY,DATASET_ROOT)

In [ ]:
# Reuse runtime TRAIN cache or extract it once with task progress.
protocol=json.loads(Path(CONFIG).read_text())
dataset=protocol['datasets'][DATASET_KEY]
cache=Path(FEATURE_CACHE_ROOT)/DATASET_KEY
if not (cache/'train.pt').is_file():
    command=[sys.executable,'-u','tools/experiment_runner.py','--extract-features-only','--extract-train-only','--root',DATASET_ROOT,'--backbone-checkpoint',CHECKPOINT_PATH,'--backbone-checkpoint-size','346284714','--backbone-checkpoint-sha256',protocol['backbone']['checkpoint_sha256'],'--feature-cache-dir',str(cache),'--output-dir','/content/unused_mars_phase1c','--dataset',dataset['dataset'],'--model-name',protocol['backbone']['model_name'],'--data-augmentation','vit','--seed','2025','--num-classes',str(dataset['num_classes']),'--num-tasks','10','--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS)]
    print('TRAIN FEATURE EXTRACTION START. Wait for task progress.',flush=True)
    subprocess.run(command,check=True)
else: print('Using existing runtime TRAIN cache:',cache)
assert (cache/'train.pt').is_file() and (cache/'metadata.json').is_file()
assert not (cache/'test.pt').exists(),'FAIL: test.pt became visible'
print('TRAIN CACHE READY | test.pt absent')

In [ ]:
# Geometry, aggregate-state and tiny fidelity-runner correctness gate.
tests=['tests/test_mars_soho_math.py','tests/test_mars_soho_learner.py','tests/test_mars_soho_phase1.py','tests/test_mars_soho_phase1b.py','tests/test_mars_soho_tangent.py','tests/test_mars_soho_phase1c.py']
completed=subprocess.run([sys.executable,'-B','-m','pytest','-q',*tests],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
print(completed.stdout,flush=True)
assert completed.returncode==0,'Correctness gate failed.'
print('MARS-SOHO PHASE-1C CORRECTNESS GATE: PASS')

## Locked fidelity study
Nine inner rank units run first, followed by twelve outer method/replicate units. `SKETCH class=i/100` is live progress inside tangent fitting. `DONE` means the unit is safely resumable on the current runtime disk. Accuracy never selects rank; only inner `G,Q` fidelity does.

In [ ]:
# Start/resume Phase 1C. No test feature or label is opened.
command=[sys.executable,'-u',RUNNER,'--config',CONFIG,'--dataset-key',DATASET_KEY,'--feature-cache-dir',str(cache),'--output-root',OUTPUT_ROOT,'--device','cuda']
print('STARTING PHASE 1C: 9 inner + 12 outer units.',flush=True)
started=time.time(); completed=subprocess.run(command)
print(f'elapsed={(time.time()-started)/60:.1f} minutes | return_code={completed.returncode}',flush=True)
assert completed.returncode==0,'Runner failed; return the traceback without changing the protocol.'
RESULT_PATH=Path(OUTPUT_ROOT)/DATASET_KEY/'phase1c_results.json'
assert RESULT_PATH.is_file()
print('PHASE-1C PROCESS COMPLETE')

In [ ]:
# Display rank selection, G/Q fidelity, accuracy and moment diagnostics.
import pandas as pd, matplotlib.pyplot as plt, seaborn as sns
payload=json.loads(RESULT_PATH.read_text())
rank_table=pd.DataFrame([{'rank':x['rank'],'inner_combined_stat_error':x['mean_inner_combined_stat_error']} for x in payload['inner_rank_selection']])
display(rank_table); print('selected rank:',payload['selected_tangent_rank'])
summary=pd.DataFrame([{'method':method,**values} for method,values in payload['outer_summary'].items()]).sort_values('combined_stat_error')
display(summary); print('gates:',json.dumps(payload['gates'],indent=2))
fig,axes=plt.subplots(1,2,figsize=(14,4))
sns.barplot(data=summary,x='method',y='combined_stat_error',ax=axes[0]); axes[0].tick_params(axis='x',rotation=30); axes[0].set_title('Lower is better: hard-WTA statistic error')
sns.barplot(data=summary,x='method',y='validation_accuracy',ax=axes[1]); axes[1].tick_params(axis='x',rotation=30); axes[1].set_title('Train-only validation Ridge accuracy')
plt.tight_layout(); plt.show()
moment_rows=[]
for method,results in payload['outer_fidelity'].items():
    for ridx,result in enumerate(results):
        if result['feature_moment_fidelity'] is not None: moment_rows.append({'method':method,'replicate':ridx,**result['feature_moment_fidelity']})
display(pd.DataFrame(moment_rows))

In [ ]:
# Export evidence only; frozen per-sample feature cache is excluded.
from google.colab import files
evidence=Path(OUTPUT_ROOT)/DATASET_KEY
shutil.copy2(CONFIG,evidence/'locked_config.json')
shutil.copy2(RUNNER,evidence/'locked_runner.py')
for relative in ['methods/mars_soho/tangent.py','methods/mars_soho/reconstruction.py','methods/mars_soho/learner.py']:
    shutil.copy2(relative,evidence/Path(relative).name)
archive=shutil.make_archive('/content/mars_soho_phase1c_cifar100_train_only','zip',root_dir=evidence)
print('artifact:',archive,'bytes=',Path(archive).stat().st_size,'sha256=',sha(archive))
files.download(archive)